# Notebook 15 — Grounded RAG evaluation (offline)

**One complete, reproducible reporting section.** This notebook scores **saved `/rag/ask` predictions** and imports no transformer/LLM. Do not treat blank human-review labels as verified; synthetic smoke-test questions used in Phase 14 are **not** a held-out benchmark. Read [Phase 15 protocol](../docs/phase_15_rag_evaluation.md) first.

In [ ]:
from pathlib import Path
from pprint import pprint
from greek_nlp.evaluation.rag_eval import load_cases, evaluate, markdown_report

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INPUT = ROOT / "results" / "phase15_local" / "predictions.json"
REPORT_DIR = ROOT / "results" / "phase15_local" / "report"
print("Input:", INPUT)
if not INPUT.is_file():
    print("NO SAVED PREDICTIONS: prepare a private dataset and collect /rag/ask results first.")

## 1. Validate saved evidence and labels

No input file is silently created and **no network calls** occur. Predictions must have the Phase 15 JSON schema.

In [ ]:
cases = load_cases(INPUT) if INPUT.is_file() else []
print("Cases:", len(cases))
print("Predictions present:", sum(c.get("prediction") is not None for c in cases))
print("Reviewer-attributed judgments:", sum(c.get("human_review") is not None for c in cases))

## 2. Aggregate metrics — with explicit denominators

Syntax-only citation checks **do not** establish faithfulness. Unanswerable-case checks require independent answerability labels; human metrics require actual dated reviews.

In [ ]:
metrics = evaluate(cases)
pprint(metrics["counts"])
pprint(metrics["automatic"])
pprint(metrics["answerability"])
pprint(metrics["human_review"])

## 3. Save aggregate report (optional)

Nothing is published automatically. Reports contain aggregates, not copyrighted news passages. Review data remain private.

In [ ]:
SAVE_REPORT = False  # Explicitly opt in after reviewing the counts.
if SAVE_REPORT and cases:
    from greek_nlp.evaluation.rag_eval import _atomic_write
    import json
    _atomic_write(REPORT_DIR / "metrics.json", json.dumps(metrics, ensure_ascii=False, indent=2) + "\n")
    _atomic_write(REPORT_DIR / "report.md", markdown_report(metrics))
    print("Saved aggregate report:", REPORT_DIR)
else:
    print("No report written. Set SAVE_REPORT=True after reviewing labels.")

## 4. Interpretation and publication checklist

- Freeze question set and record hashes, API/model/index versions, and query sampling.
- Collect answerable **and** unanswerable question predictions, with resume-safe CPU calls.
- Manually verify retrieved evidence, correctness, hallucinations and abstention cases.
- Export/import the human review CSV as described in the Phase 15 guide.
- Keep `faithfulness_verified=false` in the API; the notebook does not certify live answers.
- **Do not tick Phase 15 complete** until the planned evaluation and quality gates are actually met.